# LightGBM + Flight Distance Target Encoding

Проверяем один новый признак: безопасный OOF target encoding точного Flight Distance.

Внешние пять фолдов остаются прежними. Внутри каждого train-фолда TargetEncoder создаёт cross-fitted значения. Validation и test кодируются только статистиками внешнего train.

In [1]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"
submissions_dir = project_root / "submissions"

train_df = pd.read_parquet(processed_data_dir / "train_with_folds.parquet")
test_df = pd.read_parquet(processed_data_dir / "test_clean.parquet")
sample_submission = pd.read_csv(raw_data_dir / "sample_submission.csv")

print("train:", train_df.shape)
print("test:", test_df.shape)
print("fold sizes:", train_df["fold"].value_counts().sort_index().to_dict())

train: (699635, 25)
test: (299844, 23)
fold sizes: {0: 139927, 1: 139927, 2: 139927, 3: 139927, 4: 139927}


In [3]:
route_column = "Flight Distance"

train_routes = set(train_df[route_column])
test_routes = set(test_df[route_column])
route_counts = train_df[route_column].value_counts()

print("unique train routes:", len(train_routes))
print("unique test routes:", len(test_routes))
print("shared routes:", len(train_routes & test_routes))
print("known test rows:", test_df[route_column].isin(train_routes).mean())
print("median rows per route:", route_counts.median())
print("singleton routes:", int((route_counts == 1).sum()))

unique train routes: 3474
unique test routes: 3317
shared routes: 3270
known test rows: 0.9997965608783234
median rows per route: 52.0
singleton routes: 153


## Защита от утечки

Кодировщик создаётся заново для каждого внешнего фолда. fit_transform применяется только к train, transform — к validation и test.

In [4]:
target = "satisfaction"

categories = sorted(
    set(train_df["Class"].dropna().unique())
    | set(test_df["Class"].dropna().unique())
)
train_df["Class"] = pd.Categorical(train_df["Class"], categories=categories)
test_df["Class"] = pd.Categorical(test_df["Class"], categories=categories)

feature_columns = [
    column
    for column in train_df.columns
    if column not in [target, "id", "fold"]
]

parameters = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": 4,
    "verbosity": -1,
}

print("base features:", len(feature_columns))
print("new feature: Flight Distance TE")

base features: 22
new feature: Flight Distance TE


In [5]:
oof_predictions = np.zeros(len(train_df), dtype=np.float32)
test_predictions = np.zeros(len(test_df), dtype=np.float32)
fold_scores = []
best_iterations = []

oof_dir.mkdir(parents=True, exist_ok=True)
test_predictions_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
submissions_dir.mkdir(parents=True, exist_ok=True)

for fold in range(5):
    print(f"\nfold {fold}")

    train_mask = train_df["fold"] != fold
    valid_mask = train_df["fold"] == fold

    x_train = train_df.loc[train_mask, feature_columns].copy()
    x_valid = train_df.loc[valid_mask, feature_columns].copy()
    x_test = test_df[feature_columns].copy()

    y_train = train_df.loc[train_mask, target]
    y_valid = train_df.loc[valid_mask, target]

    inner_folds = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42 + fold,
    )

    encoder = TargetEncoder(
        target_type="binary",
        smooth="auto",
        cv=inner_folds,
    )

    x_train["Flight Distance TE"] = encoder.fit_transform(
        x_train[[route_column]],
        y_train,
    ).ravel()

    x_valid["Flight Distance TE"] = encoder.transform(
        x_valid[[route_column]]
    ).ravel()

    x_test["Flight Distance TE"] = encoder.transform(
        x_test[[route_column]]
    ).ravel()

    model = lgb.LGBMClassifier(**parameters)
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(100),
        ],
    )

    valid_prediction = model.predict_proba(
        x_valid,
        num_iteration=model.best_iteration_,
    )[:, 1]
    test_fold_prediction = model.predict_proba(
        x_test,
        num_iteration=model.best_iteration_,
    )[:, 1]

    oof_predictions[valid_mask] = valid_prediction
    test_predictions += test_fold_prediction.astype(np.float32) / 5

    fold_auc = roc_auc_score(y_valid, valid_prediction)
    fold_scores.append(float(fold_auc))
    best_iterations.append(int(model.best_iteration_))

    print("ROC AUC:", fold_auc)
    print("best iteration:", model.best_iteration_)

    np.save(oof_dir / "lightgbm_route_te_partial.npy", oof_predictions)
    np.save(
        test_predictions_dir / "lightgbm_route_te_partial.npy",
        test_predictions,
    )

    del model, encoder
    del x_train, x_valid, x_test
    del y_train, y_valid
    del valid_prediction, test_fold_prediction
    gc.collect()


fold 0


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.955681	valid_0's binary_logloss: 0.247417
[200]	valid_0's auc: 0.958464	valid_0's binary_logloss: 0.232149
[300]	valid_0's auc: 0.95939	valid_0's binary_logloss: 0.228733
[400]	valid_0's auc: 0.959704	valid_0's binary_logloss: 0.227638
[500]	valid_0's auc: 0.959859	valid_0's binary_logloss: 0.227066
[600]	valid_0's auc: 0.959986	valid_0's binary_logloss: 0.226674
[700]	valid_0's auc: 0.960052	valid_0's binary_logloss: 0.22646
[800]	valid_0's auc: 0.96014	valid_0's binary_logloss: 0.226245
[900]	valid_0's auc: 0.960182	valid_0's binary_logloss: 0.226063
[1000]	valid_0's auc: 0.960225	valid_0's binary_logloss: 0.22596
[1100]	valid_0's auc: 0.960251	valid_0's binary_logloss: 0.225865
[1200]	valid_0's auc: 0.960303	valid_0's binary_logloss: 0.225717
[1300]	valid_0's auc: 0.960311	valid_0's binary_logloss: 0.225624
[1400]	valid_0's auc: 0.960338	valid_0's binary_logloss: 0.225541
[1500]	valid_0's auc: 0.960

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.954201	valid_0's binary_logloss: 0.249541
[200]	valid_0's auc: 0.957141	valid_0's binary_logloss: 0.234815
[300]	valid_0's auc: 0.95805	valid_0's binary_logloss: 0.231598
[400]	valid_0's auc: 0.958331	valid_0's binary_logloss: 0.230527
[500]	valid_0's auc: 0.958524	valid_0's binary_logloss: 0.229836
[600]	valid_0's auc: 0.958622	valid_0's binary_logloss: 0.229517
[700]	valid_0's auc: 0.958704	valid_0's binary_logloss: 0.229289
[800]	valid_0's auc: 0.958768	valid_0's binary_logloss: 0.229126
[900]	valid_0's auc: 0.958794	valid_0's binary_logloss: 0.229002
[1000]	valid_0's auc: 0.958826	valid_0's binary_logloss: 0.228869
[1100]	valid_0's auc: 0.958829	valid_0's binary_logloss: 0.228788
Early stopping, best iteration is:
[1013]	valid_0's auc: 0.958837	valid_0's binary_logloss: 0.228844
ROC AUC: 0.9588368924864543
best iteration: 1013

fold 2


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.956017	valid_0's binary_logloss: 0.245495
[200]	valid_0's auc: 0.958868	valid_0's binary_logloss: 0.229758
[300]	valid_0's auc: 0.959806	valid_0's binary_logloss: 0.226376
[400]	valid_0's auc: 0.960133	valid_0's binary_logloss: 0.225281
[500]	valid_0's auc: 0.960298	valid_0's binary_logloss: 0.224719
[600]	valid_0's auc: 0.960431	valid_0's binary_logloss: 0.224339
[700]	valid_0's auc: 0.960492	valid_0's binary_logloss: 0.224157
[800]	valid_0's auc: 0.960546	valid_0's binary_logloss: 0.223992
[900]	valid_0's auc: 0.9606	valid_0's binary_logloss: 0.223836
[1000]	valid_0's auc: 0.960655	valid_0's binary_logloss: 0.223689
[1100]	valid_0's auc: 0.960707	valid_0's binary_logloss: 0.223561
[1200]	valid_0's auc: 0.960738	valid_0's binary_logloss: 0.223479
[1300]	valid_0's auc: 0.960753	valid_0's binary_logloss: 0.223383
[1400]	valid_0's auc: 0.960758	valid_0's binary_logloss: 0.223328
[1500]	valid_0's auc: 0.9

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.955374	valid_0's binary_logloss: 0.247127
[200]	valid_0's auc: 0.958507	valid_0's binary_logloss: 0.231768
[300]	valid_0's auc: 0.959465	valid_0's binary_logloss: 0.228313
[400]	valid_0's auc: 0.959764	valid_0's binary_logloss: 0.227182
[500]	valid_0's auc: 0.959908	valid_0's binary_logloss: 0.226564
[600]	valid_0's auc: 0.959996	valid_0's binary_logloss: 0.226209
[700]	valid_0's auc: 0.960044	valid_0's binary_logloss: 0.225975
[800]	valid_0's auc: 0.960087	valid_0's binary_logloss: 0.22578
[900]	valid_0's auc: 0.960131	valid_0's binary_logloss: 0.225645
[1000]	valid_0's auc: 0.960192	valid_0's binary_logloss: 0.225494
[1100]	valid_0's auc: 0.960223	valid_0's binary_logloss: 0.225401
[1200]	valid_0's auc: 0.960234	valid_0's binary_logloss: 0.225326
[1300]	valid_0's auc: 0.960259	valid_0's binary_logloss: 0.225242
[1400]	valid_0's auc: 0.960264	valid_0's binary_logloss: 0.225205
[1500]	valid_0's auc: 0.

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.955609	valid_0's binary_logloss: 0.247257
[200]	valid_0's auc: 0.958605	valid_0's binary_logloss: 0.23178
[300]	valid_0's auc: 0.959498	valid_0's binary_logloss: 0.228325
[400]	valid_0's auc: 0.959779	valid_0's binary_logloss: 0.227336
[500]	valid_0's auc: 0.95994	valid_0's binary_logloss: 0.226827
[600]	valid_0's auc: 0.960021	valid_0's binary_logloss: 0.226566
[700]	valid_0's auc: 0.960075	valid_0's binary_logloss: 0.226273
[800]	valid_0's auc: 0.960114	valid_0's binary_logloss: 0.226106
[900]	valid_0's auc: 0.960135	valid_0's binary_logloss: 0.226002
[1000]	valid_0's auc: 0.960149	valid_0's binary_logloss: 0.225916
[1100]	valid_0's auc: 0.960162	valid_0's binary_logloss: 0.225836
[1200]	valid_0's auc: 0.960167	valid_0's binary_logloss: 0.225748
[1300]	valid_0's auc: 0.960165	valid_0's binary_logloss: 0.225709
[1400]	valid_0's auc: 0.960188	valid_0's binary_logloss: 0.225614
[1500]	valid_0's auc: 0.9

In [6]:
oof_auc = roc_auc_score(train_df[target], oof_predictions)

np.save(oof_dir / "lightgbm_route_te.npy", oof_predictions)
np.save(test_predictions_dir / "lightgbm_route_te.npy", test_predictions)

metrics = {
    "parameters": parameters,
    "feature": "Flight Distance Target Encoding",
    "target_encoder_smooth": "auto",
    "target_encoder_cv": 5,
    "fold_auc": fold_scores,
    "mean_fold_auc": float(np.mean(fold_scores)),
    "oof_auc": float(oof_auc),
    "best_iterations": best_iterations,
}

with (metrics_dir / "lightgbm_route_te.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(metrics, file, indent=4)

submission = sample_submission.copy()
submission["satisfaction"] = test_predictions
submission.to_csv(
    submissions_dir / "lightgbm_route_te_submission.csv",
    index=False,
)

print("fold AUC:", fold_scores)
print("mean fold AUC:", np.mean(fold_scores))
print("OOF ROC AUC:", oof_auc)

baseline_path = oof_dir / "lightgbm.npy"
if baseline_path.exists():
    baseline_predictions = np.load(baseline_path)
    baseline_auc = roc_auc_score(train_df[target], baseline_predictions)
    print("baseline LightGBM:", baseline_auc)
    print("TE improvement:", oof_auc - baseline_auc)

print("results saved")

fold AUC: [0.9603531995583718, 0.9588368924864543, 0.9608820613035495, 0.9603249147336925, 0.9602052661607016]
mean fold AUC: 0.960120466848554
OOF ROC AUC: 0.9601191652066354
baseline LightGBM: 0.9588781741592428
TE improvement: 0.0012409910473926145
results saved
